# Colab bootstrap (run at the start of EVERY session)
Runtime -> Change runtime type -> GPU. Each teammate has their own runtime/GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/traffic'   # shared folder: data/bdd5.zip, runs/, results/
!mkdir -p $DRIVE/runs $DRIVE/results

In [ ]:
REPO = 'https://github.com/<your-org>/traffic-detection.git'   # <- edit
!git clone $REPO /content/repo 2>/dev/null || (cd /content/repo && git pull)
%cd /content/repo
!pip install -q ultralytics albumentations pycocotools   # do NOT reinstall torch on Colab

In [ ]:
# Train from LOCAL disk: reading thousands of small files from Drive is very slow
!mkdir -p /content/data && unzip -q -n $DRIVE/data/bdd5.zip -d /content/data
!python scripts/verify_data.py --data_root /content/data/bdd5 --n_vis 4

## Run one of these (each teammate runs their own model in their own session)

In [ ]:
# Faster R-CNN (add --resume after a disconnect)
!python scripts/train_frcnn.py --data_root /content/data/bdd5 --out $DRIVE/runs/frcnn --epochs 20 --batch 8

In [ ]:
# YOLOv8s (use --resume after a disconnect; --mosaic 0.0 --name yolov8s_640_nomosaic for the ablation)
!python scripts/train_yolo.py --data_root /content/data/bdd5 --project $DRIVE/runs --name yolov8s_640 --epochs 50

In [ ]:
# Final evaluation on TEST (only after training is finished)
!python scripts/evaluate.py --data_root /content/data/bdd5 --split test --name frcnn --model frcnn --ckpt $DRIVE/runs/frcnn/best.pt --out $DRIVE/results
!python scripts/evaluate.py --data_root /content/data/bdd5 --split test --name yolov8s --model yolo --ckpt $DRIVE/runs/yolov8s_640/weights/best.pt --out $DRIVE/results